#  Notebook 4 — Modelo con Mitigación de Sesgo
## HMDA New York 2024: Sesgo, Equidad y Explicabilidad en Hipotecas

**Autores:** Izan Cuesta Corbí · Dennis García Solera · Marcos Segurado Llopis · Jesús Cano Moya  
**Dataset:** Home Mortgage Disclosure Act (HMDA) — New York 2024 (CFPB)

---
> **Objetivo de este notebook:** Mitigar el sesgo demográfico del modelo LightGBM ganador mediante técnicas algorítmicas (*Fairlearn*). Evaluaremos el *trade-off* resultante entre rendimiento y equidad, y auditaremos con SHAP cómo se neutraliza la discriminación en las nuevas predicciones.

> `RANDOM_STATE = 42` fijado globalmente para reproducibilidad.

--- 

### Imports

In [15]:
import re
import pandas as pd
import joblib

RANDOM_STATE = 42

---
---

## Carga de Datos y Modelo Ganador

NO MODIFICADO:

En este notebook hemos cargado los conjuntos de entrenamiento y test generados en el Notebook 1, ya preprocesados, imputados y escalados. Al partir de los mismos datos que el baseline, garantizamos que cualquier diferencia en los resultados sea atribuible exclusivamente al modelo utilizado y no al preprocesamiento.

---

### Carga y limpieza del dataset

NOTA: LIMPIEZA LIGHTGBM

In [16]:
def limpiar_columnas(df):
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    return df

In [22]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

train = limpiar_columnas(train)
test  = limpiar_columnas(test)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

train.head()

Train: (226665, 81)
Test:  (56667, 81)


,derived_msa_md,county_code,preapproval,loan_type,loan_purpose,lien_status,reverse_mortgage,open_end_line_of_credit,business_or_commercial_purpose,loan_amount,...,derived_race_Black_or_African_American,derived_race_Free_Form_Text_Only,derived_race_Joint,derived_race_Native_Hawaiian_or_Other_Pacific_Islander,derived_race_Race_Not_Available,derived_race_White,derived_sex_Female,derived_sex_Joint,derived_sex_Male,derived_sex_Sex_Not_Available
0,-0.053043,-0.136364,0.0,0.0,9.666667,1.0,0.0,1.0,0.0,-0.030303,...,False,False,False,False,False,True,False,True,False,False
1,5.598696,0.772727,0.0,0.0,-0.333333,0.0,0.0,0.0,0.0,-0.303030,...,False,False,False,False,False,True,False,False,True,False
2,-1.898609,-1.318182,0.0,0.0,0.666667,1.0,0.0,0.0,0.0,-0.545455,...,False,False,False,False,False,True,False,True,False,False
3,0.414435,-0.227273,0.0,0.0,-0.333333,0.0,0.0,0.0,1.0,-0.272727,...,False,False,False,False,True,False,False,False,False,True
4,0.000000,0.363636,0.0,0.0,9.666667,0.0,0.0,0.0,1.0,0.696970,...,False,False,False,False,True,False,True,False,False,False


---
### Selección de Columnas

NO MODIFICADO: 

A diferencia del Notebook 2, en este caso hemos incluido todas las columnas disponibles como features, incluyendo los atributos sensibles (`derived_race_*`, `derived_sex_*`, `derived_ethnicity_*`). El objetivo es evaluar el comportamiento natural de los modelos de ensamble sin ninguna restricción, para identificar el sesgo inherente que introducen cuando tienen acceso a información demográfica.

In [23]:
SENSITIVE_COLS = [col for col in train.columns if 
                  col.startswith('derived_race_') or 
                  col.startswith('derived_sex_') or 
                  col.startswith('derived_ethnicity_')]

FEATURE_COLS = [col for col in train.columns if col != 'action_taken']

print(f"Nº de features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")
print(f"\nNº de features sensibles: {len(SENSITIVE_COLS)}")
print(f"Features: {SENSITIVE_COLS}")

Nº de features: 80
Features: ['derived_msa_md', 'county_code', 'preapproval', 'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage', 'open_end_line_of_credit', 'business_or_commercial_purpose', 'loan_amount', 'loan_term', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'other_nonamortizing_features', 'construction_method', 'occupancy_type', 'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest', 'total_units', 'income', 'debt_to_income_ratio', 'applicant_credit_score_type', 'co_applicant_credit_score_type', 'applicant_ethnicity_1', 'co_applicant_ethnicity_1', 'applicant_ethnicity_observed', 'co_applicant_ethnicity_observed', 'applicant_race_1', 'co_applicant_race_1', 'applicant_race_observed', 'co_applicant_race_observed', 'applicant_sex', 'co_applicant_sex', 'applicant_sex_observed', 'co_applicant_sex_observed', 'applicant_age', 'applicant_age_above_62', 'submission_of_application', 'initially_payable_to_institution', 'aus_1

---
### Separación de Train/Test

In [12]:
X_train = train[FEATURE_COLS] # columnas que entran al modelo 
y_train = train['action_taken'] # columna que queremos predecir

X_test = test[FEATURE_COLS] 
y_test = test['action_taken']

# Guardamos los grupos sensibles por separado (para usarlo en fairness)
groups_train = train[SENSITIVE_COLS]
groups_test  = test[SENSITIVE_COLS]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\ngroups_train: {groups_train.shape}")
print(f"groups_test:  {groups_test.shape}")

X_train: (226665, 80)
X_test:  (56667, 80)

groups_train: (226665, 18)
groups_test:  (56667, 18)


---
### Mejor Modelo: LightGBM

In [14]:
modelo = joblib.load('../models/lightgbm.pkl')

---
---